# Testing linear attention blocks

Get root.

In [ ]:
import os, sys, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import importlib
from torch.nn.attention.flex_attention import flex_attention
from pathlib import Path
!pip install linear-attention-transformer

# Working repo
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")
root = Path("/content/mini-gLM")
if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url])
sys.path.insert(0, str(root))

# Torch configs
torch.manual_seed(111)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
flex_attention = torch.compile(flex_attention, dynamic = True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Synthetic data.

In [ ]:
# Configs
B = 8
L = 200
D = 256
vocab_size = 512

# Synthetic data
x = torch.randint(low = 0, high = 512, size = (B, L)).to(device)
embed = torch.randn(B, L, D).float().to(device)
probs = torch.full((B, L), 0.5).to(device)
mask = torch.bernoulli(probs).bool().to(device)

Test dense attention transformer.

In [ ]:
from src.transformer import SimpleTransformer

# Model
model = SimpleTransformer(
    d_model = D,
    num_heads = 8,
    p_drop = 0.1
).to(device)


model(embed, attn_mask = mask).shape

Test linear attention transformer.

In [ ]:
import src.transformer; importlib.reload(src.transformer)
from src.transformer import LinearTransformer

class LinearGLM(nn.Module):
    def __init__(
        self, 
        num_blocks: int,
        d_model: int,
        num_heads: int,
        p_drop: float,
        vocab_size: int,
        max_seq_len: int
    ):     
        super().__init__()
        self.num_blocks = num_blocks
        self.d_model = d_model
        self.num_heads = num_heads
        self.p_drop = p_drop
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        
        self.model = nn.ModuleList([
            LinearTransformer(
                d_model = d_model,
                num_heads = num_heads,
                p_drop = p_drop,
                vocab_size = vocab_size,
                max_seq_len = max_seq_len
            ) for _ in range(num_blocks)
        ])
        self.final = nn.Linear(d_model, vocab_size)

    def forward(self, x, attn_mask):
        for block in self.model:
            x = block(x, attn_mask)
        logits = self.final(x)
        return logits

Try to integrate alternating dense and linear attention blocks.